# Localization evaluation in stages

**Question:** does localization improve as the update moves through the original, online-GNN, and post-transition-GNN representations?

- **Original:** source GNN checkpoint on the original online graph.
- **Online GNN:** saved embeddings from the GNN trained during the update.
- **Post-transition GNN:** final GNN applied once to a rebuilt reference+update graph. Reference nodes use the raw pre-GNN features that trained this checkpoint; update nodes use post-transition AE encodings.

Each stage gets the **same reference-only localization MLP protocol**: same labeled bootstrap scans, coordinates, architecture, training settings, and seed. The MLP is fit separately for each stage because each stage has its own embedding space. The held-out update coordinates are used only for scoring. All reported position errors are in the original coordinate units.

**Primary result:** compare paired held-out 2D RMSE by stage within each run. Lower is better. The per-scan table shows whether any aggregate change is hiding regressions on individual scans.

In [1]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from ae import encode_prepared_dataset
from batch_transition import project_prepared_batch_to_vocabulary
from gufu_data import load_prepared_dataset
from gufu_evaluation import train_reference_location_mlp_predictions
from lp_streaming import LinkPredictionConfig, predict_edges_streaming
from spatial_normalization import load_spatial_transform
from streaming_gnn import (
    infer_streaming_gnn_checkpoint_embeddings,
    infer_streaming_gnn_embeddings,
    load_streaming_gnn_state,
)
from streaming_update import build_streaming_online_graph, load_streaming_online_graph

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "gufu.py").is_file():
    raise RuntimeError("Run this notebook with the GUFU repository root as the working directory.")

OUTPUT_ROOTS = {
    "GF_from1F_12": PROJECT_ROOT / "output_from1F_12",
    "GF_fromGF_12": PROJECT_ROOT / "output_fromGF_12",
}
DEVICE_OVERRIDE = None  # None uses the device recorded in each run's manifest.
STAGES = ("original", "online_gnn", "post_transition_gnn")


In [2]:
def _latest_snapshot(output_root: Path) -> tuple[Path, dict]:
    run_roots = sorted((output_root / "notebook_runs" / "GF").glob("real-GF-*"))
    if len(run_roots) != 1:
        raise ValueError(f"Expected one GF run under {output_root}, found {len(run_roots)}")

    completed = []
    for manifest_path in (run_roots[0] / "streaming_pipeline" / "snapshots").glob("*/manifest.json"):
        snapshot_dir = manifest_path.parent
        if not (snapshot_dir / "COMPLETED.json").is_file():
            continue
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        completed.append((manifest["sequence_number"], snapshot_dir, manifest))
    if not completed:
        raise FileNotFoundError(f"No completed streaming snapshot found under {output_root}")
    _, snapshot_dir, manifest = max(completed, key=lambda item: item[0])
    return snapshot_dir, manifest, run_roots[0]


def _split_pair_embeddings(pair_embeddings: np.ndarray, reference_count: int):
    return pair_embeddings[:reference_count], pair_embeddings[reference_count:]


def evaluate_run(run_name: str, output_root: Path):
    snapshot_dir, snapshot, run_root = _latest_snapshot(output_root)
    config = snapshot["pipeline_config"]
    device = DEVICE_OVERRIDE or config["device"]
    update_batch = load_prepared_dataset(snapshot_dir / "input_prepared_batch.json")
    update_ids = tuple(scan.scan_id for scan in update_batch.scans)
    if update_ids != tuple(snapshot["update_scan_ids"]):
        raise ValueError(f"{run_name}: update scan order differs from the snapshot manifest")

    # The original graph contains both bootstrap references and this update batch.
    original_graph_dir = snapshot_dir / "online_graph"
    original_graph = load_streaming_online_graph(original_graph_dir)
    online_state = load_streaming_gnn_state(
        snapshot_dir / "online_gnn", original_graph_dir, device=device
    )
    reference_ids = tuple(original_graph.manifest.labeled_reference_ids)
    if tuple(original_graph.manifest.update_sample_ids) != update_ids:
        raise ValueError(f"{run_name}: original graph update IDs do not match the evaluation batch")
    original_rows = {node_id: row for row, node_id in enumerate(original_graph.manifest.node_ids)}
    reference_rows = [original_rows[scan_id] for scan_id in reference_ids]
    update_rows = [original_rows[scan_id] for scan_id in update_ids]

    # Stage 1: run the source checkpoint over the original graph.
    source_embeddings = infer_streaming_gnn_checkpoint_embeddings(
        original_graph_dir,
        snapshot_dir / "source_gnn_checkpoint.pt",
        online_state.manifest.model_config_record,
        reference_rows + update_rows,
        expected_checkpoint_sha256=online_state.manifest.source_gnn_checkpoint_sha256,
        chunk_size=config["gnn_chunk_size"],
        device=device,
        temporary_parent=output_root,
    )
    original_reference, original_queries = _split_pair_embeddings(
        source_embeddings, len(reference_rows)
    )

    # Stage 2: use the online GNN's saved embeddings on that same graph.
    online_reference = np.asarray(online_state.node_embeddings[reference_rows]).copy()
    online_queries = np.asarray(online_state.node_embeddings[update_rows]).copy()

    # Load the final reference graph/checkpoint and verify bootstrap labels and spatial scale.
    current = snapshot["current_state"]
    final_reference_graph_dir = snapshot_dir / current["reference_graph"]
    final_reference_graph = load_streaming_online_graph(final_reference_graph_dir)
    final_state = load_streaming_gnn_state(
        snapshot_dir / current["reference_gnn"], final_reference_graph_dir, device=device
    )
    final_reference_ids = tuple(final_reference_graph.manifest.labeled_reference_ids)
    original_coordinates = dict(zip(
        reference_ids, original_graph.manifest.labeled_reference_coordinates
    ))
    final_coordinates = dict(zip(
        final_reference_ids, final_reference_graph.manifest.labeled_reference_coordinates
    ))
    if original_coordinates != final_coordinates or set(reference_ids) != set(final_reference_ids):
        raise ValueError(f"{run_name}: bootstrap coordinate labels changed across stages")

    original_build = original_graph.manifest.graph_build_config
    final_build = final_reference_graph.manifest.graph_build_config
    # Resolve the transform under this output root; the path embedded in the
    # training manifest may still begin with the old directory name ("output/").
    transform_path = run_root / "artifacts" / "spatial" / f"{snapshot['dataset_id']}.json"
    print(f"{run_name}: spatial transform = {transform_path}")
    transform = load_spatial_transform(transform_path)
    if transform.transform_sha256 != original_build.spatial_transform_sha256:
        raise ValueError(f"{run_name}: original graph spatial-transform identity is inconsistent")
    if final_build.spatial_transform_sha256 != transform.transform_sha256:
        raise ValueError(f"{run_name}: spatial transform changed across stages")
    reference_xy = np.asarray([original_coordinates[scan_id] for scan_id in reference_ids])

    # Stage 3: rebuild the transient post-transition query graph, matching production evaluation.
    with tempfile.TemporaryDirectory(prefix=f"{run_name}-stages-") as temporary_name:
        temporary_dir = Path(temporary_name)
        projected_batch = project_prepared_batch_to_vocabulary(
            update_batch, tuple(snapshot["vocabulary_after"])
        )
        update_encoder = encode_prepared_dataset(
            projected_batch,
            snapshot_dir / current["ae_checkpoint"],
            temporary_dir / "updated_query_encoder.json",
            batch_size=config["encoder_batch_size"],
            device=device,
        )
        lp_result_dir = temporary_dir / "updated_reference_lp_result"
        predict_edges_streaming(
            snapshot_dir / current["reference_lp_input"],
            lp_result_dir,
            LinkPredictionConfig.model_validate(config["link_prediction"]),
        )
        query_graph_dir = temporary_dir / "post_transition_query_graph"
        build_streaming_online_graph(
            snapshot_dir / current["reference_lp_input"],
            lp_result_dir,
            projected_batch,
            update_encoder,
            query_graph_dir,
            virtual_edge_sigma=config["graph_build"]["virtual_edge_sigma"],
            spatial_transform_path_override=transform_path,
            reference_node_ids_override=final_reference_graph.manifest.node_ids,
            reference_node_features_override=final_reference_graph.node_features,
            num_workers=config["rssi_pair_num_workers"],
        )
        query_graph = load_streaming_online_graph(query_graph_dir)
        if tuple(query_graph.manifest.update_sample_ids) != update_ids:
            raise ValueError(f"{run_name}: rebuilt query graph update IDs do not match")
        query_rows_by_id = {
            node_id: row for row, node_id in enumerate(query_graph.manifest.node_ids)
        }
        final_reference_rows = [query_rows_by_id[scan_id] for scan_id in final_reference_ids]
        final_update_rows = [query_rows_by_id[scan_id] for scan_id in update_ids]
        final_pair_embeddings = infer_streaming_gnn_embeddings(
            query_graph_dir,
            final_state.model,
            final_reference_rows + final_update_rows,
            chunk_size=config["gnn_chunk_size"],
            device=device,
            temporary_parent=output_root,
        )
    final_reference, final_queries = _split_pair_embeddings(
        final_pair_embeddings, len(final_reference_rows)
    )
    final_reference_xy = np.asarray([final_coordinates[scan_id] for scan_id in final_reference_ids])

    # Fit the same reference-only probe recipe independently in each representation space.
    probe = config["fingerprint_update"]
    predictions = {}
    for stage, reference_embeddings, query_embeddings, coordinates in (
        ("original", original_reference, original_queries, reference_xy),
        ("online_gnn", online_reference, online_queries, reference_xy),
        ("post_transition_gnn", final_reference, final_queries, final_reference_xy),
    ):
        predictions[stage] = train_reference_location_mlp_predictions(
            reference_embeddings,
            coordinates,
            query_embeddings,
            spatial_transform=transform,
            hidden_dim=probe["hidden_dim"],
            epochs=probe["updater_epochs"],
            learning_rate=probe["updater_learning_rate"],
            seed=probe["seed"],
            device=device,
        )

    truth_by_id = {
        item.scan_id: np.asarray([item.x, item.y], dtype=np.float64)
        for item in update_batch.evaluation_positions
    }
    paired_ids = [scan_id for scan_id in update_ids if scan_id in truth_by_id]
    if not paired_ids:
        raise ValueError(f"{run_name}: no held-out update coordinates are available")
    prediction_by_stage = {
        stage: dict(zip(update_ids, values)) for stage, values in predictions.items()
    }
    errors = {
        stage: np.asarray([
            np.linalg.norm(np.asarray(prediction_by_stage[stage][scan_id]) - truth_by_id[scan_id])
            for scan_id in paired_ids
        ])
        for stage in STAGES
    }

    per_scan = pd.DataFrame({
        "run": run_name,
        "scan_id": paired_ids,
        "truth_x": [truth_by_id[scan_id][0] for scan_id in paired_ids],
        "truth_y": [truth_by_id[scan_id][1] for scan_id in paired_ids],
    })
    for stage in STAGES:
        stage_predictions = np.asarray([prediction_by_stage[stage][scan_id] for scan_id in paired_ids])
        per_scan[f"{stage}_prediction_x"] = stage_predictions[:, 0]
        per_scan[f"{stage}_prediction_y"] = stage_predictions[:, 1]
        per_scan[f"{stage}_error"] = errors[stage]
    per_scan["online_minus_original_error"] = per_scan["online_gnn_error"] - per_scan["original_error"]
    per_scan["post_transition_minus_original_error"] = (
        per_scan["post_transition_gnn_error"] - per_scan["original_error"]
    )

    original_rmse = float(np.sqrt(np.mean(errors["original"] ** 2)))
    summary = []
    for stage in STAGES:
        radial = errors[stage]
        rmse = float(np.sqrt(np.mean(radial ** 2)))
        summary.append({
            "run": run_name,
            "stage": stage,
            "held_out_scans": len(paired_ids),
            "2D_RMSE": rmse,
            "RMSE_delta_vs_original": rmse - original_rmse,
            "RMSE_change_percent_vs_original": (100.0 * (rmse / original_rmse - 1.0)) if original_rmse else np.nan,
            "mean_radial_error": float(np.mean(radial)),
            "median_radial_error": float(np.median(radial)),
            "p90_radial_error": float(np.percentile(radial, 90)),
            "scans_improved_vs_original": int(np.sum(radial < errors["original"])) if stage != "original" else 0,
            "scans_worsened_vs_original": int(np.sum(radial > errors["original"])) if stage != "original" else 0,
        })
    return summary, per_scan


The post-transition stage reruns link prediction and rebuilds the reference+query graph. It supplies the final reference graph’s raw node features so both reference and update nodes receive exactly one pass through the final GNN. This can take a while, especially for the larger `GF_fromGF_12` run.


In [3]:
summary_rows = []
per_scan_frames = []
for run_name, output_root in OUTPUT_ROOTS.items():
    print(f"Evaluating {run_name}...")
    run_summary, run_per_scan = evaluate_run(run_name, output_root)
    summary_rows.extend(run_summary)
    per_scan_frames.append(run_per_scan)

summary = pd.DataFrame(summary_rows)
per_scan = pd.concat(per_scan_frames, ignore_index=True)
summary["_stage_order"] = summary["stage"].map({stage: i for i, stage in enumerate(STAGES)})
summary = summary.sort_values(["run", "_stage_order"]).drop(columns="_stage_order").reset_index(drop=True)

display(summary)

Evaluating GF_from1F_12...
GF_from1F_12: spatial transform = /home/oem/GUFU/output_from1F_12/notebook_runs/GF/real-GF-fingerprint0-ae-v2-mse-rssi-pairs-v1-spatial-v2-softplus-cosine-t0p1-nw1-terminal-node-v1-d256-z32-drop0p1-dataf939a3b408f75d37/artifacts/spatial/GF.json
Evaluating GF_fromGF_12...
GF_fromGF_12: spatial transform = /home/oem/GUFU/output_fromGF_12/notebook_runs/GF/real-GF-fingerprint0-ae-v2-mse-rssi-pairs-v1-spatial-v2-softplus-cosine-t0p1-nw1-terminal-node-v1-d256-z32-drop0p1-data1be80e24f66a5214/artifacts/spatial/GF.json


,run,stage,held_out_scans,2D_RMSE,RMSE_delta_vs_original,RMSE_change_percent_vs_original,mean_radial_error,median_radial_error,p90_radial_error,scans_improved_vs_original,scans_worsened_vs_original
0,GF_from1F_12,original,419,282.116392,0.000000,0.000000,236.257375,241.779725,440.166119,0,0
1,GF_from1F_12,online_gnn,419,44.630586,-237.485805,-84.180080,38.989564,35.325529,68.651411,384,35
2,GF_from1F_12,post_transition_gnn,419,55.735150,-226.381241,-80.243916,46.911075,40.700288,93.845506,377,42
3,GF_fromGF_12,original,1733,192.583959,0.000000,0.000000,172.193409,163.806916,294.918182,0,0
4,GF_fromGF_12,online_gnn,1733,148.706869,-43.877090,-22.783356,119.504818,97.194235,230.949078,1163,570
5,GF_fromGF_12,post_transition_gnn,1733,100.608550,-91.975409,-47.758603,82.570071,73.964328,155.260018,1380,353


## Inspect the paired scan-level changes

A negative delta means the later-stage prediction error is lower than the original-stage error for that same held-out scan. Start with the largest regressions; then inspect the full `per_scan` table if needed.

In [4]:
worst_online = per_scan.sort_values("online_minus_original_error", ascending=False).groupby("run", as_index=False).head(10)
worst_post_transition = per_scan.sort_values("post_transition_minus_original_error", ascending=False).groupby("run", as_index=False).head(10)

display(worst_online[["run", "scan_id", "original_error", "online_gnn_error", "online_minus_original_error"]])
display(worst_post_transition[["run", "scan_id", "original_error", "post_transition_gnn_error", "post_transition_minus_original_error"]])

,run,scan_id,original_error,online_gnn_error,online_minus_original_error
1033,GF_fromGF_12,scan:99d28e27354c9e73340db4aad8aef3330a9803077...,177.140881,440.804486,263.663605
764,GF_fromGF_12,scan:a6c7ff390593e0b6a628a1db322066ad172314ebe...,50.103275,308.848690,258.745415
1018,GF_fromGF_12,scan:c395577c122de9662c60683b3b6ab666881002ba8...,186.751344,440.299067,253.547723
1030,GF_fromGF_12,scan:f5bac89950be8da77f09ff0637c5b9819fa2457ff...,183.154437,435.031701,251.877263
1007,GF_fromGF_12,scan:8ff3386f43e240bf3aff850ef6cb6f2a2d6b0e5c8...,150.782934,399.003721,248.220787
1767,GF_fromGF_12,scan:ff679dd60e9f0c6708ec15425fc38730888000c71...,151.364105,399.038566,247.674461
1042,GF_fromGF_12,scan:737d76dd0e0ec52bfae9d6c86435e8d998f44e87c...,191.509748,439.166240,247.656492
1020,GF_fromGF_12,scan:088d5d211f4a47b26fbc1920d86e64de53e085835...,166.184074,413.423033,247.238959
1021,GF_fromGF_12,scan:717902b1705a39380260378fd6a09eeb85b6c3e95...,182.761252,426.537364,243.776112
1016,GF_fromGF_12,scan:67f431db7b173a4e7f1423062908b9f46ee2115a0...,186.060963,427.992331,241.931368


,run,scan_id,original_error,post_transition_gnn_error,post_transition_minus_original_error
971,GF_fromGF_12,scan:8b67462f9b12446a5717d8fcb2de30c445d3a3d66...,71.109161,384.224757,313.115596
596,GF_fromGF_12,scan:bdf67498121003dde1142e6b5a1979907db58d40c...,84.527537,383.828335,299.300798
967,GF_fromGF_12,scan:da0e3003a5854c34ba6c6912852ba1e8dbcb56a2e...,114.875226,405.005990,290.130763
589,GF_fromGF_12,scan:a20345c90ef695690feb670fdadb15fe895a4bd55...,16.254864,304.111320,287.856456
613,GF_fromGF_12,scan:ee94b24a4d036e4adb8958bbf0ea1ff94dc9afb66...,68.993428,322.965657,253.972228
595,GF_fromGF_12,scan:00a75cf490420f3c9ab712c36bbb1bac32efc749b...,58.876429,297.442683,238.566254
973,GF_fromGF_12,scan:4fb137ad3a95173c522ce628ba663e06993b705b3...,162.255700,400.320944,238.065244
976,GF_fromGF_12,scan:ea3a0f5e2c47c9969f37846d41ccd3fe2cb6eb858...,137.400496,372.728405,235.327910
977,GF_fromGF_12,scan:3c46e861f6f6e9883f0833577a0cc7d5f8fe0b6d2...,90.657302,323.993224,233.335921
602,GF_fromGF_12,scan:0dfc0e5b18ce1bbbdf0b5f7022a242b41fbf0e433...,64.590866,288.378951,223.788085
